# Concurrent LLM Sessions with Shared Working Memory

Reproducible benchmark: sequential vs continuous batching vs prefix-cacheable shared context vs mid-prompt shared context.

**Runtime: T4 GPU.** Synthetic prompts only. Results are uploaded to 0x0.st and printed below.

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
!pip install -q -U vllm transformers httpx

In [ ]:
!wget -q https://raw.githubusercontent.com/mahithpaulin/shared-context-lab/main/bench.py -O /content/bench.py
print(open('/content/bench.py').read()[:200])

In [ ]:
!cd /content && python bench.py 2>&1 | tee /content/bench.log

In [ ]:
import json, subprocess, os
p='/content/results.json'
if not os.path.exists(p):
    print('results.json missing; inspect the run cell above')
else:
    r=json.load(open(p))
    print('model:', r['meta']['model'], '| vllm:', r['meta']['vllm'])
    print('gpu:', r['meta']['gpu'], '| peak MiB:', r.get('gpu_peak_mib'))
    print(f"{'apc':5} {'layout':7} {'mode':11} {'wall_s':>8} {'tok/s':>8} {'ttft50':>8} {'ttft95':>8} {'match':>6}")
    for run in r['runs']:
        a=run['aggregate']
        print(f"{str(run['apc']):5} {run['layout']:7} {run['concurrency']:11} {a['wall_s']:8.2f} {a['throughput_tok_s']:8.1f} {a['ttft_p50_s']:8.3f} {a['ttft_p95_s']:8.3f} {run['exact_match_vs_baseline']:6.2f}")
    for f in ('/content/results.json','/content/bench.log'):
        if os.path.exists(f):
            out=subprocess.run(['curl','-sS','-F',f'file=@{f}','https://0x0.st'],capture_output=True,text=True,timeout=120).stdout.strip()
            print('ARTIFACT', f, '->', out)

## Reading the result

- `prefix` + APC should cut TTFT for every session after the first because the shared block is computed once.
- `middle` holds the same information but is not a common prefix, so APC should not help. That gap is the case for position-independent KV reuse.
- `concurrent` should raise throughput and shift TTFT; isolation must remain 100%.